# House Prices — Full Encoding Techniques Comparison
Applies every method from the reference "encoding techniques" notebook to the
Ames Housing dataset (`train.csv` / `test.csv`, target = `SalePrice`).

- Method 1: Label Encoding
- Method 2: One-Hot Encoding
- Method 3: Feature Hashing
- Method 4: Encoding categories with dataset statistics (frequency encoding)
- Cyclic features: `MoSold` (month sold) via sin/cos
- Method 5: Target Encoding
- Method 6: K-Fold Target Encoding
- Summary: model performance across all methods

Since `SalePrice` is continuous, this uses **RMSE** and **R²** instead of accuracy,
and a single model type (Random Forest) across every method so the comparison is fair.

## Import libraries

In [2]:
import glob
import time

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction import FeatureHasher
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

## Load the data

In [3]:
def find_csv(keyword):
    candidates = glob.glob(f'*{keyword}*.csv') + glob.glob(f'**/*{keyword}*.csv', recursive=True)
    if not candidates:
        raise FileNotFoundError(f"No CSV file matching '*{keyword}*.csv' found in {glob.os.getcwd()}")
    return candidates[0]

train_path = find_csv('train')
test_path = find_csv('test')
print('Using train file:', train_path)
print('Using test file:', test_path)

df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print('train data set has got {} rows and {} columns'.format(df_train.shape[0], df_train.shape[1]))
print('test data set has got {} rows and {} columns'.format(df_test.shape[0], df_test.shape[1]))

Using train file: train (1).csv
Using test file: test (1).csv
train data set has got 1460 rows and 81 columns
test data set has got 1459 rows and 80 columns


## Define train and target

In [4]:
X = df_train.drop(['SalePrice', 'Id'], axis=1).reset_index(drop=True)
y = df_train['SalePrice'].reset_index(drop=True)

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X[cat_cols] = X[cat_cols].fillna('None')
for c in num_cols:
    X[c] = X[c].fillna(X[c].median())

print('categorical columns:', len(cat_cols))
print('numeric columns:', len(num_cols))
print('remaining NaNs:', X.isna().sum().sum())

categorical columns: 43
numeric columns: 36
remaining NaNs: 0


## Evaluation helper
Same model, same split, every time — the only thing that changes between
methods is how the categorical columns get encoded.

In [5]:
results = {}

def evaluate(X_encoded, y, name):
    start = time.time()
    X_tr, X_val, y_tr, y_val = train_test_split(X_encoded, y, random_state=42, test_size=0.2)
    model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    elapsed = time.time() - start
    results[name] = {'RMSE': rmse, 'R2': r2, 'seconds': elapsed}
    print(f'{name:>28s}   RMSE: {rmse:>10,.0f}   R2: {r2:.4f}   time: {elapsed:.2f}s')
    return model

## Method 1: Label Encoding
Every category gets substituted with an integer — simple, but it invents a
false sense of order/magnitude between categories that don't actually have one.

In [6]:
%%time

X_label = X.copy()
for c in cat_cols:
    X_label[c] = LabelEncoder().fit_transform(X_label[c])

print('label-encoded data set has got {} rows and {} columns'.format(*X_label.shape))
X_label.head(3)

label-encoded data set has got 1460 rows and 79 columns
CPU times: user 15.7 ms, sys: 982 µs, total: 16.7 ms
Wall time: 15.2 ms


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,3,65.0,8450,1,1,3,3,0,4,...,0,0,3,4,1,0,2,2008,8,4
1,20,3,80.0,9600,1,1,3,3,0,2,...,0,0,3,4,1,0,5,2007,8,4
2,60,3,68.0,11250,1,1,0,3,0,4,...,0,0,3,4,1,0,9,2008,8,4


In [7]:
evaluate(X_label, y, 'Label Encoding')

              Label Encoding   RMSE:     28,402   R2: 0.8948   time: 0.36s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

## Method 2: One-Hot Encoding
Each category becomes its own binary column via `pd.get_dummies()`.

In [8]:
%%time

X_ohe = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print('one-hot encoded data set has got {} rows and {} columns'.format(*X_ohe.shape))
X_ohe.head(3)

one-hot encoded data set has got 1460 rows and 260 columns
CPU times: user 91.6 ms, sys: 2.99 ms, total: 94.6 ms
Wall time: 93.2 ms


,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,60,65.0,8450,7,5,2003,2003,196.0,706,0,...,False,False,False,False,True,False,False,False,True,False
1,20,80.0,9600,6,8,1976,1976,0.0,978,0,...,False,False,False,False,True,False,False,False,True,False
2,60,68.0,11250,7,5,2001,2002,162.0,486,0,...,False,False,False,False,True,False,False,False,True,False


In [9]:
evaluate(X_ohe, y, 'One-Hot Encoding')

            One-Hot Encoding   RMSE:     29,026   R2: 0.8902   time: 0.52s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

## Method 3: Feature Hashing
Hashes every column's value (as a string) into a fixed number of buckets —
here 512 — instead of one column per unique category. Cheaper than one-hot on
high-cardinality columns, at the cost of occasional hash collisions.

In [10]:
%%time

X_hash_str = X.astype(str)
hasher = FeatureHasher(n_features=512, input_type='string')
X_hashed = hasher.transform(X_hash_str.values)

print('hashed data set has got {} rows and {} columns'.format(*X_hashed.shape))

hashed data set has got 1460 rows and 512 columns
CPU times: user 37.7 ms, sys: 3.02 ms, total: 40.7 ms
Wall time: 37.8 ms


In [11]:
evaluate(X_hashed, y, 'Feature Hashing')

             Feature Hashing   RMSE:     42,352   R2: 0.7661   time: 1.40s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

## Method 4: Encoding categories with dataset statistics
Every category is replaced with how often it appears in the training set —
categories that behave similarly (e.g. two common neighborhoods) end up with
similar encoded values.

In [12]:
%%time

X_stat = X.copy()
for c in cat_cols:
    counts = X_stat[c].value_counts()
    X_stat[c] = X_stat[c].map(counts)

print('dataset-statistics encoded data set has got {} rows and {} columns'.format(*X_stat.shape))
X_stat.head(3)

dataset-statistics encoded data set has got 1460 rows and 79 columns
CPU times: user 26.5 ms, sys: 2.99 ms, total: 29.5 ms
Wall time: 27.7 ms


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,1151,65.0,8450,1454,1369,925,1311,1459,1052,...,0,0,1453,1179,1406,0,2,2008,1267,1198
1,20,1151,80.0,9600,1454,1369,925,1311,1459,47,...,0,0,1453,1179,1406,0,5,2007,1267,1198
2,60,1151,68.0,11250,1454,1369,484,1311,1459,1052,...,0,0,1453,1179,1406,0,9,2008,1267,1198


In [13]:
evaluate(X_stat, y, 'Dataset Statistics (Frequency)')

Dataset Statistics (Frequency)   RMSE:     28,402   R2: 0.8948   time: 0.44s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

## Encoding cyclic features
`MoSold` (month the house was sold) is cyclic — month 12 and month 1 are
actually adjacent, which a plain integer doesn't capture. Sine/cosine
transforms preserve that wraparound; the rest of the categorical columns are
still one-hot encoded.

In [14]:
%%time

X_cyclic = X.copy()
month_max = X_cyclic['MoSold'].max()
X_cyclic['MoSold_sin'] = np.sin(2 * np.pi * X_cyclic['MoSold'] / month_max)
X_cyclic['MoSold_cos'] = np.cos(2 * np.pi * X_cyclic['MoSold'] / month_max)
X_cyclic = X_cyclic.drop('MoSold', axis=1)

X_cyclic_encoded = pd.get_dummies(X_cyclic, columns=cat_cols, drop_first=True)

print('cyclic + one-hot encoded data set has got {} rows and {} columns'.format(*X_cyclic_encoded.shape))
X_cyclic_encoded[['MoSold_sin', 'MoSold_cos']].head(3)

cyclic + one-hot encoded data set has got 1460 rows and 261 columns
CPU times: user 24.8 ms, sys: 1.02 ms, total: 25.8 ms
Wall time: 24.7 ms


,MoSold_sin,MoSold_cos
0,0.866025,5.000000e-01
1,0.500000,-8.660254e-01
2,-1.000000,-1.836970e-16


In [15]:
evaluate(X_cyclic_encoded, y, 'Cyclic (MoSold) + One-Hot')

   Cyclic (MoSold) + One-Hot   RMSE:     28,916   R2: 0.8910   time: 0.53s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

## Method 5: Target Encoding
Every category gets replaced with the **average `SalePrice`** for that
category. This version computes the averages on the full dataset before
splitting — which leaks a bit of validation-set information into the
encoding and tends to make the validation score look better than it really
is. Method 6 fixes that.

In [16]:
%%time

tmp = X.copy()
tmp['SalePrice'] = y

X_target = X.copy()
for c in cat_cols:
    means = tmp.groupby(c)['SalePrice'].mean()
    X_target[c] = X_target[c].map(means)

print('target-encoded data set has got {} rows and {} columns'.format(*X_target.shape))
X_target.head(3)

target-encoded data set has got 1460 rows and 79 columns
CPU times: user 34.5 ms, sys: 4 ms, total: 38.5 ms
Wall time: 37.3 ms


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,191004.994787,65.0,8450,181130.538514,183452.131483,164754.818378,180183.746758,180950.95682,176938.047529,...,0,0,180404.663455,187596.837998,182046.410384,0,2,2008,173401.836622,175202.219533
1,20,191004.994787,80.0,9600,181130.538514,183452.131483,164754.818378,180183.746758,180950.95682,177934.574468,...,0,0,180404.663455,187596.837998,182046.410384,0,5,2007,173401.836622,175202.219533
2,60,191004.994787,68.0,11250,181130.538514,183452.131483,206101.665289,180183.746758,180950.95682,176938.047529,...,0,0,180404.663455,187596.837998,182046.410384,0,9,2008,173401.836622,175202.219533


In [17]:
evaluate(X_target, y, 'Target Encoding (leaky)')

     Target Encoding (leaky)   RMSE:     28,390   R2: 0.8949   time: 0.41s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

### K-Fold Target Encoding
To reduce the overfitting from Method 5, each row's encoding is computed
**only from the other folds**, never from its own fold — so no row ever sees
its own target value (or that of rows sharing its fold) baked into its
encoding.

In [18]:
%%time

X_fold = X.copy()
for c in cat_cols:
    X_fold[c] = np.nan

kf = KFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in kf.split(X):
    fold_train = X.iloc[train_idx].copy()
    fold_train['SalePrice'] = y.iloc[train_idx]
    for c in cat_cols:
        means = fold_train.groupby(c)['SalePrice'].mean()
        X_fold.loc[val_idx, c] = X.iloc[val_idx][c].map(means).values

global_mean = y.mean()
for c in cat_cols:
    X_fold[c] = X_fold[c].fillna(global_mean)

print('k-fold target-encoded data set has got {} rows and {} columns'.format(*X_fold.shape))
X_fold.head(3)

k-fold target-encoded data set has got 1460 rows and 79 columns
CPU times: user 362 ms, sys: 2.98 ms, total: 365 ms
Wall time: 366 ms


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,60,191344.810811,65.0,8450,181305.792777,183493.489517,164117.602721,180488.933270,181104.263699,177327.317102,...,0,0,180441.316695,187624.658824,182461.821174,0,2,2008,172987.037475,175044.121849
1,20,191907.323465,80.0,9600,181349.259673,183459.214026,163739.561995,180477.258126,181119.221937,169477.777778,...,0,0,180624.254296,188240.670873,182427.226749,0,5,2007,172883.104167,174563.507788
2,60,191344.810811,68.0,11250,181305.792777,183493.489517,207354.167939,180488.933270,181104.263699,177327.317102,...,0,0,180441.316695,187624.658824,182461.821174,0,9,2008,172987.037475,175044.121849


In [19]:
evaluate(X_fold, y, 'K-Fold Target Encoding')

      K-Fold Target Encoding   RMSE:     28,952   R2: 0.8907   time: 0.52s


RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

## Summary of model performance

In [20]:
summary = pd.DataFrame(results).T
summary = summary[['RMSE', 'R2', 'seconds']].sort_values('RMSE')
summary

,RMSE,R2,seconds
Target Encoding (leaky),28390.320732,0.894918,0.409688
Label Encoding,28401.772482,0.894834,0.355975
Dataset Statistics (Frequency),28402.050234,0.894832,0.440407
Cyclic (MoSold) + One-Hot,28916.473840,0.890987,0.527479
K-Fold Target Encoding,28952.178097,0.890718,0.515629
One-Hot Encoding,29025.575619,0.890163,0.523058
Feature Hashing,42352.278663,0.766149,1.400063


In [23]:

X_test = df_test.drop(['Id'], axis=1).copy()

cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_test[cat_cols] = X_test[cat_cols].fillna('None')
for c in num_cols:
    X_test[c] = X_test[c].fillna(X[c].median())


X_full = pd.concat([X, X_test], axis=0)
X_test_encoded = X_test.copy()
X_train_encoded = X.copy()

for c in cat_cols:
    le = LabelEncoder()
    le.fit(X_full[c])
    X_train_encoded[c] = le.transform(X_train_encoded[c])
    X_test_encoded[c] = le.transform(X_test_encoded[c])


final_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
final_model.fit(X_train_encoded, y)

# 3. Generate test set predictions
y_pred = final_model.predict(X_test_encoded)

# 4. Save Submission File
submission = pd.DataFrame({'Id': df_test['Id'], 'SalePrice': y_pred})
submission.to_csv('submission.csv', index=False)
print("submission.csv successfully created with shape:", submission.shape)

submission.csv successfully created with shape: (1459, 2)


### Notes
- All methods share the same model (`RandomForestRegressor`, 200 trees) and the
  same train/validation split, so the differences you see come from the
  encoding alone.
- **Target Encoding (leaky)** vs **K-Fold Target Encoding** is the most
  instructive comparison here — watch how much the score drops once the
  leakage is removed. That gap is the real lesson of this method.
- **Feature Hashing** hashes numeric columns as strings too, the same way the
  original reference notebook does — it throws away their magnitude, which
  is a real limitation on a dataset like this one that's mostly numeric.
- For a dataset this size, One-Hot Encoding is a perfectly reasonable default;
  Feature Hashing and count-based statistics start to earn their keep on much
  higher-cardinality categorical columns than are found here.